In [5]:
import sys
from pathlib import Path

sys.path.append(str(Path.cwd().parent / "functions"))

In [6]:
from signal_processing import load_raw_gdf, rename_and_montage, bandpass_filter, run_ica, apply_ica, epoch_raw, compute_tfr, get_motor_events
from dataset import log_normalize, build_and_save_dataset, log_transform, EEGDataset
from train import run_loso_optimized, calibrate_subject

import matplotlib.pyplot as plt
import numpy as np
import mne

In [7]:
gdf_path_1 = Path.cwd().parent / "data" / "BCICIV_2a_gdf" / "A01T.gdf"
gdf_path_2 = Path.cwd().parent / "data" / "BCICIV_2a_gdf" / "A02T.gdf"
gdf_path_3 = Path.cwd().parent / "data" / "BCICIV_2a_gdf" / "A03T.gdf"

In [8]:
exclude_map = {
    1: [0],
    2: [0],
    3: [0]
}

In [9]:
outdir = "../data/processed_ICA_1-3"
build_and_save_dataset([1,2,3], outdir, exclude_map=exclude_map)

processing subject 1...
Extracting GDF parameters from /Users/nathanielrogers/Library/CloudStorage/OneDrive-NorthwesternUniversity/Year 4/Medical Makers/eeg-wheelchair-control/data/BCICIV_2a_gdf/A01T.gdf...
Setting channel info structure...
Could not determine channel type of the following channels, they will be set as EEG:
EEG-Fz, EEG, EEG, EEG, EEG, EEG, EEG, EEG-C3, EEG, EEG-Cz, EEG, EEG-C4, EEG, EEG, EEG, EEG, EEG, EEG, EEG, EEG-Pz, EEG, EEG
Creating raw.info structure...
Reading 0 ... 672527  =      0.000 ...  2690.108 secs...


/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/contextlib.py:144: RuntimeWarning: Channel names are not unique, found duplicates for: {'EEG'}. Applying running numbers for duplicates.
  next(self.gen)


Filtering raw data in 1 contiguous segment
Setting up band-pass filter from 3 - 35 Hz

IIR filter parameters
---------------------
Butterworth bandpass zero-phase (two-pass forward and reverse) non-causal filter:
- Filter order 20 (effective, after forward-backward)
- Cutoffs at 3.00, 35.00 Hz: -6.02, -6.02 dB



/Users/nathanielrogers/Library/CloudStorage/OneDrive-NorthwesternUniversity/Year 4/Medical Makers/eeg-wheelchair-control/functions/signal_processing.py:50: FutureWarning: Montage name 'standard_1020' is deprecated and will be removed in MNE 1.14. Use 'colin27_1020' instead.
  montage = mne.channels.make_standard_montage('standard_1020')


Use rng= instead of random_state= in new code
Fitting ICA to data using 22 channels (please be patient, this may take a while)
Selecting by number: 20 components
Fitting ICA took 8.3s.
Using EOG channels: EOG-left, EOG-central, EOG-right
... filtering ICA sources
Setting up band-pass filter from 1 - 10 Hz

FIR filter parameters
---------------------
Designing a two-pass forward and reverse, zero-phase, non-causal bandpass filter:
- Windowed frequency-domain design (firwin2) method
- Hann window
- Lower passband edge: 1.00
- Lower transition bandwidth: 0.50 Hz (-12 dB cutoff frequency: 0.75 Hz)
- Upper passband edge: 10.00 Hz
- Upper transition bandwidth: 0.50 Hz (-12 dB cutoff frequency: 10.25 Hz)
- Filter length: 2500 samples (10.000 s)

... filtering target
Setting up band-pass filter from 1 - 10 Hz

FIR filter parameters
---------------------
Designing a two-pass forward and reverse, zero-phase, non-causal bandpass filter:
- Windowed frequency-domain design (firwin2) method
- Hann w

/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/contextlib.py:144: RuntimeWarning: Channel names are not unique, found duplicates for: {'EEG'}. Applying running numbers for duplicates.
  next(self.gen)


... (3 outputs left out)

/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/contextlib.py:144: RuntimeWarning: Channel names are not unique, found duplicates for: {'EEG'}. Applying running numbers for duplicates.
  next(self.gen)


Filtering raw data in 1 contiguous segment
Setting up band-pass filter from 3 - 35 Hz

IIR filter parameters
---------------------
Butterworth bandpass zero-phase (two-pass forward and reverse) non-causal filter:
- Filter order 20 (effective, after forward-backward)
- Cutoffs at 3.00, 35.00 Hz: -6.02, -6.02 dB



/Users/nathanielrogers/Library/CloudStorage/OneDrive-NorthwesternUniversity/Year 4/Medical Makers/eeg-wheelchair-control/functions/signal_processing.py:50: FutureWarning: Montage name 'standard_1020' is deprecated and will be removed in MNE 1.14. Use 'colin27_1020' instead.
  montage = mne.channels.make_standard_montage('standard_1020')


Use rng= instead of random_state= in new code
Fitting ICA to data using 22 channels (please be patient, this may take a while)
Selecting by number: 20 components
Fitting ICA took 6.5s.
Using EOG channels: EOG-left, EOG-central, EOG-right
... filtering ICA sources
Setting up band-pass filter from 1 - 10 Hz

FIR filter parameters
---------------------
Designing a two-pass forward and reverse, zero-phase, non-causal bandpass filter:
- Windowed frequency-domain design (firwin2) method
- Hann window
- Lower passband edge: 1.00
- Lower transition bandwidth: 0.50 Hz (-12 dB cutoff frequency: 0.75 Hz)
- Upper passband edge: 10.00 Hz
- Upper transition bandwidth: 0.50 Hz (-12 dB cutoff frequency: 10.25 Hz)
- Filter length: 2500 samples (10.000 s)

... filtering target
Setting up band-pass filter from 1 - 10 Hz

FIR filter parameters
---------------------
Designing a two-pass forward and reverse, zero-phase, non-causal bandpass filter:
- Windowed frequency-domain design (firwin2) method
- Hann w

In [12]:
subject_ids = [1, 2, 3]
accuracies, mean = run_loso_optimized(subject_ids, input_type="spec",n_epochs=50, batch_size=32, data_dir="../data/processed_ICA_1-3")

/Users/nathanielrogers/Library/CloudStorage/OneDrive-NorthwesternUniversity/Year 4/Medical Makers/eeg-wheelchair-control/functions/dataset.py:356: UserWarning: The given NumPy array is not writable, and PyTorch does not support non-writable tensors. This means writing to this tensor will result in undefined behavior. You may want to copy the array to protect its data or make it writable before converting it to a tensor. This type of warning will be suppressed for the rest of this program. (Triggered internally at /Users/runner/work/pytorch/pytorch/torch/csrc/utils/tensor_numpy.cpp:214.)
  x = torch.from_numpy(self.X_list[subject_idx][local_idx]).float()


[Subject 1] Epoch 1/50 | Train Loss: 1.4596 | Val Loss: 1.4091
[Subject 1] Epoch 2/50 | Train Loss: 1.4050 | Val Loss: 1.3932
[Subject 1] Epoch 3/50 | Train Loss: 1.3652 | Val Loss: 1.3821
[Subject 1] Epoch 4/50 | Train Loss: 1.3777 | Val Loss: 1.3745
[Subject 1] Epoch 5/50 | Train Loss: 1.3117 | Val Loss: 1.3526
[Subject 1] Epoch 6/50 | Train Loss: 1.3158 | Val Loss: 1.3497
[Subject 1] Epoch 7/50 | Train Loss: 1.2956 | Val Loss: 1.3373
[Subject 1] Epoch 8/50 | Train Loss: 1.2888 | Val Loss: 1.3216
[Subject 1] Epoch 9/50 | Train Loss: 1.2822 | Val Loss: 1.3137
[Subject 1] Epoch 10/50 | Train Loss: 1.2649 | Val Loss: 1.2808
[Subject 1] Epoch 11/50 | Train Loss: 1.2142 | Val Loss: 1.2785
[Subject 1] Epoch 12/50 | Train Loss: 1.2148 | Val Loss: 1.2683
[Subject 1] Epoch 13/50 | Train Loss: 1.1651 | Val Loss: 1.2606
[Subject 1] Epoch 14/50 | Train Loss: 1.1751 | Val Loss: 1.2425
[Subject 1] Epoch 15/50 | Train Loss: 1.1317 | Val Loss: 1.2358
[Subject 1] Epoch 16/50 | Train Loss: 1.1207 | Va

In [ ]:
for s in [1, 2, 3]:
    y = np.load(f"../data/processed_ICA_1-3/subject_{s}_y.npy")
    print(f"Subject {s}: {np.unique(y, return_counts=True)}")

In [ ]:
X = np.load("../data/processed_ICA_1-3/subject_1_X.npy")
y = np.load("../data/processed_ICA_1-3/subject_1_y.npy")

import matplotlib.pyplot as plt
fig, axes = plt.subplots(4, 3, figsize=(12, 10))
for class_idx in range(4):
    samples = X[y == class_idx][:3]  # 3 samples per class
    for i, sample in enumerate(samples):
        axes[class_idx, i].imshow(sample[7], aspect='auto', origin='lower')  # channel C3
        axes[class_idx, i].set_title(f"Class {class_idx} sample {i}")
plt.tight_layout()
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(16, 4))
for class_idx in range(4):
    samples = X[y == class_idx]
    mean_tfr = samples[:, 7, :, :].mean(axis=0)  # avg over all trials, channel C3
    axes[class_idx].imshow(mean_tfr, aspect='auto', origin='lower')
    axes[class_idx].set_title(f"Class {class_idx} mean")
plt.tight_layout()
plt.show()

In [ ]:
exclude_map = {}
build_and_save_dataset([1,2,3,4,5,6], "../data/processed_global_norm", exclude_map=exclude_map)


In [ ]:
subject_ids = [1, 2, 3, 4, 5, 6]
accuracies, mean = run_loso_optimized(subject_ids, n_epochs=50, batch_size=32, data_dir="../data/processed_global_norm")

In [ ]:
exclude_map = {}
build_and_save_dataset([1,2,3], "../data/processed_global_norm", exclude_map=exclude_map)


In [ ]:
subject_ids = [1, 2, 3]
accuracies, mean = run_loso_optimized(subject_ids, "spec", n_epochs=50, batch_size=32, data_dir="../data/processed_global_norm")

In [ ]:
subject_ids = [1, 2, 3, 4, 5, 6]
accuracies, mean = run_loso_optimized(subject_ids, "spec", n_epochs=50, batch_size=32, data_dir="../data/processed_global_norm")

In [ ]:
calibrate_subject(1, "spec")

In [ ]:
exclude_map = {}
build_and_save_dataset([7,8,9], "../data/processed_global_norm", exclude_map=exclude_map)

In [ ]:
subject_ids = [1, 2, 3, 4, 5, 6, 7, 8, 9]
accuracies, mean = run_loso_optimized(subject_ids, "spec", n_epochs=50, batch_size=32, data_dir="../data/processed_global_norm")

In [ ]:
for i in range(1, 10):
    x = calibrate_subject(i, "spec")
    print(f"Calibratied Accuracy: {x}")
    print("---------------------------BREAK-------------------------")